In [0]:
import requests
import json
import pandas as pd

url = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson"
response = requests.get(url)
data = response.json()

# تحويل البيانات الخام إلى Spark DataFrame
json_strings = [json.dumps(f) for f in data['features']]
new_raw_df = spark.createDataFrame([(s,) for s in json_strings], ["raw_content"])

In [0]:
# الإضافة هنا تكون تراكمية (Append) لبناء السجل التاريخي
new_raw_df.write.format("delta").mode("append").saveAsTable("default.earthquakes_bronze")
print(f"Bronze Updated: {new_raw_df.count()} records added.")

In [0]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType
from delta.tables import DeltaTable

# تعريف الـ Schema
schema = StructType([
    StructField("id", StringType()),
    StructField("properties", StructType([
        StructField("mag", DoubleType()),
        StructField("place", StringType()),
        StructField("time", LongType())
    ]))
])

# تجهيز البيانات الجديدة مع حذف التكرار اللحظي
updates_df = spark.table("default.earthquakes_bronze") \
    .withColumn("parsed", from_json(col("raw_content"), schema)) \
    .select(col("parsed.id"), col("parsed.properties.mag").alias("magnitude"), 
            col("parsed.properties.place").alias("location"), 
            col("parsed.properties.time").alias("timestamp_ms")) \
    .filter("id IS NOT NULL") \
    .dropDuplicates(["id"]) # حماية إضافية من التكرار

# التأكد من وجود الجدول قبل عمل Merge، إذا لم يوجد ننشئه (Initialize)
if not spark.catalog.tableExists("default.earthquakes_silver"):
    updates_df.write.format("delta").saveAsTable("default.earthquakes_silver")
else:
    silver_table = DeltaTable.forName(spark, "default.earthquakes_silver")
    silver_table.alias("target").merge(
        updates_df.alias("source"), "target.id = source.id"
    ).whenNotMatchedInsertAll().execute()

In [0]:
from pyspark.sql.functions import from_unixtime, current_timestamp

gold_df = spark.table("default.earthquakes_silver") \
    .withColumn("event_time", from_unixtime(col("timestamp_ms") / 1000)) \
    .withColumn("processed_at", current_timestamp())

gold_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("default.earthquakes_gold")

# عرض النتيجة النهائية للتأكد
display(spark.sql("SELECT count(*) as total_count FROM default.earthquakes_gold"))

In [0]:
%sql
select count(*) from earthquakes_gold;